In [1]:
import torch
import evaluate
from datasets import load_dataset, DatasetDict, concatenate_datasets, interleave_datasets
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import DataCollatorForTokenClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


## Project goal

## Loading datasets

In [2]:
# Hungarian
raw_hun_dataset = load_dataset("ficsort/SzegedNER", revision="convert/parquet")
# Multilingual
raw_multi_dataset = load_dataset("betterdataai/gliner-multilingual-ner-silver-v1")
# German
raw_ger_dataset = load_dataset("bltlab/open-ner-core-types", "GermEval_deu")

## Inspecting datasets

In [3]:
print(f"Features in raw Hungarian dataset: {raw_hun_dataset["train"].features.keys()}")
print(f"Features in raw multilingual dataset: {raw_multi_dataset["train"].features.keys()}")
print(f"Features in raw German dataset: {raw_ger_dataset["train"].features.keys()}")

Features in raw Hungarian dataset: dict_keys(['id', 'tokens', 'ner', 'document_id', 'sentence_id'])
Features in raw multilingual dataset: dict_keys(['id', 'language', 'source', 'tokenized_text', 'ner', 'text'])
Features in raw German dataset: dict_keys(['id', 'tokens', 'ner_tags'])


In [4]:
# Renaming 'tokenized_text' to 'tokens' for consistency
multi_dataset = raw_multi_dataset.rename_column("tokenized_text", "tokens")

# Renaming 'ner_labels' to 'ner' for consistency
ger_dataset = raw_ger_dataset.rename_column("ner_tags", "ner")

# Renaming dataset for consistency
hun_dataset = raw_hun_dataset

In [5]:
print(f"Hungarian dataset splits and sizes: {hun_dataset.num_rows}")
print(f"Multilingual dataset splits and sizes: {multi_dataset.num_rows}")
print(f"German dataset splits and sizes: {ger_dataset.num_rows}")

Hungarian dataset splits and sizes: {'train': 12282, 'validation': 940, 'test': 1728}
Multilingual dataset splits and sizes: {'train': 348958, 'validation': 19382, 'test': 19396}
German dataset splits and sizes: {'test': 5100, 'dev': 2200, 'train': 24000}


In [6]:
# Renaming 'dev' to 'validation' in the German dataset
if "dev" in ger_dataset:
    ger_dataset["validation"] = ger_dataset.pop("dev")

In [7]:
# Checking languages in multilingual dataset
print(multi_dataset["train"].unique("language"))
# Checking sources in multilingual dataset
print(multi_dataset["train"].unique("source"))

['English', 'Italian', 'Swedish', 'Vietnamese', 'Spanish', 'German', 'French', 'Chinese', 'Indonesian', 'Dutch', 'Japanese', 'Korean']
['PleIAs/SEC', 'HuggingFaceFW/finewiki', 'wikimedia/wikipedia', 'wikimedia/wikisource', 'allenai/c4', 'google/wiki40b', 'tylu0117/Bloomberg_Financial_News', 'MedRAG/pubmed', 'common-pile/pubmed', 'wikiann']


We don't need other languages apart from English from the multilingual dataset, as we already have a dataset in German and we can further filter it by only keeping records from specific sources.

## Creating English dataset

In [8]:
def get_dataset_with_conditions(ds, sources=None, languages=None):
    if sources is not None and ds["source"] not in sources:
        return False
    if languages is not None and ds["language"] not in languages:
        return False
        
    return True

In [9]:
# Filtering for English language and Wikipedia sources
eng_dataset = multi_dataset.filter(
    get_dataset_with_conditions, 
    fn_kwargs={"languages": ["English"], "sources": ["wikimedia/wikipedia", "wikiann"]}
)

In [10]:
print(f"Hungarian dataset splits and sizes: {hun_dataset.num_rows}")
print(f"English dataset splits and sizes: {eng_dataset.num_rows}")
print(f"German dataset splits and sizes: {ger_dataset.num_rows}")

Hungarian dataset splits and sizes: {'train': 12282, 'validation': 940, 'test': 1728}
English dataset splits and sizes: {'train': 9482, 'validation': 559, 'test': 546}
German dataset splits and sizes: {'test': 5100, 'train': 24000, 'validation': 2200}


In [11]:
print(f"Total records in Hungarian dataset: {sum(hun_dataset.num_rows.values())}")
print(f"Total records in English dataset: {sum(eng_dataset.num_rows.values())}")
print(f"Total records in German dataset: {sum(ger_dataset.num_rows.values())}")

Total records in Hungarian dataset: 14950
Total records in English dataset: 10587
Total records in German dataset: 31300


Dataset sizes differ significantly, oversampling will be needed for creating the training set.

## NER tags in datasets

In [12]:
print("Hungarian dataset NER tags sample")
[print(row, end='\n')for row in hun_dataset["train"]['ner'][:3]]
print("\n")
print("English dataset NER tags sample")
[print(row, end='\n')for row in eng_dataset["train"]['ner'][:3]]
print("\n")
print("German dataset NER tags sample")
[print(row, end='\n')for row in ger_dataset["train"]['ner'][:3]];

Hungarian dataset NER tags sample
[0, 0, 0, 3, 4, 4, 4, 0, 3, 0, 0, 0, 0, 3, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 3, 4, 0, 0, 3, 0, 0]
[0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 3, 0]
[0, 3, 4, 4, 4, 4, 0, 0, 3, 4, 4, 0, 0, 3, 4, 0, 3, 0, 0, 0, 0, 0, 3, 0, 3, 0, 0, 0]


English dataset NER tags sample
[[23, 23, 'country']]
[[0, 1, 'full_name'], [4, 6, 'date_of_birth']]
[[20, 20, 'street_address'], [45, 45, 'company'], [47, 48, 'company']]


German dataset NER tags sample
[5, 0, 0, 0, 3, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 5, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


The English dataset uses word-span annotations while the Hungarian and German dataset uses BIO tags.

Checking granularity of NER tags in the English dataset

In [13]:
# Extracting the third tag of every row and creating a set of uniqe tags
unique_eng_tags = {
    tag[2] 
    for row in eng_dataset['train']['ner'] 
    for tag in row
}

print(f"English unique tags:\n{unique_eng_tags}")

English unique tags:
{'state', 'date', 'unique_identifier', 'last_name', 'organization', 'city', 'api_key', 'gender', 'hospital_name', 'company', 'account_number', 'language', 'full_name', 'middle_name', 'political_view', 'device_id', 'name', 'time', 'age', 'work_of_art', 'job_title', 'law', 'product', 'education_level', 'first_name', 'event', 'postal_code', 'occupation', 'date_time', 'national_id', 'url', 'date_of_birth', 'county', 'street_address', 'race_ethnicity', 'facility', 'local_latlng', 'user_name', 'medical_record_number', 'country'}


In [14]:
print("Hungarian BIO tags:", hun_dataset['train'].features['ner'].feature.names)
print("German BIO tags:", ger_dataset['train'].features['ner'].feature.names)

Hungarian BIO tags: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
German BIO tags: ['O', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG', 'B-PER', 'I-PER']


Word-span annotation of the English dataset needs to be converted to BIO tags and Hungarian and German datasets uses different BIO tag ID's.

## Harmonizing NER label schemas

We must pick the schema of the Hungarian dataset as master, since we don't want to lose 'MISC' tagged words in the Hungarian text by using the German schema, and the English dataset has very granular tags which we cannot map our German and Hungarian datasets to.

In [15]:
master_label_names = hun_dataset['train'].features['ner'].feature.names
master_id2label = {i:name for i, name in enumerate(master_label_names)}
master_label2id = {name: i for i, name in enumerate(master_label_names)}

print(master_id2label)
print(master_label2id)

{0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC', 7: 'B-MISC', 8: 'I-MISC'}
{'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8}


Creating master dataset tags

In [16]:
label_mapping_dict = {
    # PERSON
    'full_name': 'PER',
    'first_name': 'PER',
    'last_name': 'PER',
    'middle_name': 'PER',
    'name': 'PER',

    # ORGANIZATION
    'company': 'ORG',
    'organization': 'ORG',
    'hospital_name': 'ORG',

    # LOCATION
    'city': 'LOC',
    'country': 'LOC',
    'state': 'LOC',
    'street_address': 'LOC',
    'postal_code': 'LOC',
    'county': 'LOC',
}

Converting character-span tags to BIO tags in English dataset

In [17]:
def ner_spans_to_bio_tags_batched(ds, ner_col, mapping_dict):
    standardized_ner_list = []
    
    for list_of_spans in ds[ner_col]:
        standardized_list = []
        for span in list_of_spans:
            if len(span) >= 3:
                start, end, label = span[0], span[1], span[2]
                if label in mapping_dict:
                    new_label = mapping_dict[label]
                    standardized_list.append([start, end, new_label])
                    
        standardized_ner_list.append(standardized_list)
        
    ds['ner'] = standardized_ner_list
    return ds

In [18]:
eng_dataset = eng_dataset.map(
    ner_spans_to_bio_tags_batched,
    batched=True,
    fn_kwargs={
        "ner_col": "ner",
        "mapping_dict": label_mapping_dict
    }
)


Map:   0%|          | 0/9482 [00:00<?, ? examples/s]

Map:   0%|          | 0/559 [00:00<?, ? examples/s]

Map:   0%|          | 0/546 [00:00<?, ? examples/s]

Verifying result

In [22]:
print(f"Transformed tags: {eng_dataset["train"]['ner'][:3]}")

ValueError: Column 'ner' doesn't exist.

In [23]:
eng_dataset = eng_dataset.rename_column("ner", "ner_spans")

ValueError: Original column name ner not in the dataset. Current columns in the dataset: ['id', 'language', 'source', 'tokens', 'ner_spans', 'text']

In [24]:
def convert_spans_to_ids_batched(ds, token_col, ner_col, master_label2id):
    all_bio_tags = []
    
    for tokens, list_of_spans in zip(ds[token_col], ds[ner_col]):
        
        # Initializing 'O' tags
        num_tokens = len(tokens)
        bio_tags = [0] * num_tokens
        
        # Mapping tags
        for span in list_of_spans:
            # Span = [start_idx, end_idx, 'label']
            start, end, label = span[0], span[1], span[2]
            
            # Inserting tags with proper prefix
            if start < num_tokens:
                bio_tags[start] = master_label2id[f"B-{label}"]
                for i in range(start + 1, min(end + 1, num_tokens)):
                    bio_tags[i] = master_label2id[f"I-{label}"]
                    
        all_bio_tags.append(bio_tags)
        
    ds['ner'] = all_bio_tags
    return ds

In [25]:
eng_dataset = eng_dataset.map(
    convert_spans_to_ids_batched,
    batched=True,
    fn_kwargs={
        "token_col":"tokens",
        "ner_col": "ner_spans",
        "master_label2id": master_label2id
        }
    )

Map:   0%|          | 0/9482 [00:00<?, ? examples/s]

Map:   0%|          | 0/559 [00:00<?, ? examples/s]

Map:   0%|          | 0/546 [00:00<?, ? examples/s]

In [26]:
print(eng_dataset["train"][1]['tokens'])
print(eng_dataset["train"][1]['ner_spans'])  
print(eng_dataset["train"][1]['ner'])  


['Marcio', 'Ravelomanantsoa', '(', 'born', '15', 'October', '1996', ')', 'is', 'a', 'Malagasy', 'football', 'striker', 'who', 'currently', 'plays', 'for', 'JET', 'Kintana', '.', 'Honours', 'Madagascar', 'African', 'Nations', 'Championship', 'third', 'place', ':', '2022', 'References', '1996', 'births', 'Living', 'people', 'People', 'from', 'Toliara', 'Malagasy', 'men', "'", 's', 'footballers', 'Madagascar', 'men', "'", 's', 'international', 'footb']
[[0, 1, 'PER']]
[1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [27]:
def visualize_ner_tags(words, ner_tags, master_id2label):
    word_line = ""
    tag_line = ""

    for word, tag in zip(words, ner_tags):
        full_label = master_id2label[tag]
        max_length = max(len(word), len(full_label))

        word_line += word + " " * (max_length - len(word) + 1)
        tag_line += full_label + " " * (max_length - len(full_label) + 1)

    print(word_line)
    print(tag_line)

In [28]:
visualize_ner_tags(
    words= eng_dataset['train'][1]['tokens'],
    ner_tags= eng_dataset['train'][1]['ner'],
    master_id2label=master_id2label
)

Marcio Ravelomanantsoa ( born 15 October 1996 ) is a Malagasy football striker who currently plays for JET Kintana . Honours Madagascar African Nations Championship third place : 2022 References 1996 births Living people People from Toliara Malagasy men ' s footballers Madagascar men ' s international footb 
B-PER  I-PER           O O    O  O       O    O O  O O        O        O       O   O         O     O   O   O       O O       O          O       O       O            O     O     O O    O          O    O      O      O      O      O    O       O        O   O O O           O          O   O O O             O     


Creating German id mapping dictionary

In [29]:
ger_labels = ger_dataset['train'].features['ner'].feature.names
ger_id2label = {id:name for id,name in enumerate(ger_labels)}

Comparing German and Master (Hungarian) id mappings

In [30]:
print(f"German id to label mapping: {ger_id2label}")
print(f"Master id to label mapping: {master_id2label}")

German id to label mapping: {0: 'O', 1: 'B-LOC', 2: 'I-LOC', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-PER', 6: 'I-PER'}
Master id to label mapping: {0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC', 7: 'B-MISC', 8: 'I-MISC'}


Mapping German dataset to the Hungarian ids

In [31]:
ger_dataset = ger_dataset.rename_column("ner", "ner_old_id")

In [32]:
def map_ger_tags_to_master(batch, ner_col, ger_id2label, master_label2id):
    all_new_tags = []
    for sentence_tags in batch[ner_col]:
        new_sentence_tags = [master_label2id[ger_id2label[tag_id]] for tag_id in sentence_tags]
        all_new_tags.append(new_sentence_tags)
    
    batch["ner"] = all_new_tags
    return batch

In [33]:
ger_dataset = ger_dataset.map(
    map_ger_tags_to_master,
    batched=True,
    fn_kwargs={
        "ner_col": "ner_old_id",
        "ger_id2label": ger_id2label,
        "master_label2id": master_label2id
    }
)

Verifying new mapping

In [34]:
print(f"Old German BIO mapping: {ger_dataset['train']['ner_old_id'][0]}")
print(f"New German BIO mapping: {ger_dataset['train']['ner'][0]}")

Old German BIO mapping: [5, 0, 0, 0, 3, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
New German BIO mapping: [1, 0, 0, 0, 3, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## Concatenating all datasets

Keeping only necessary columns for concatenation

In [35]:
def drop_cols_in_ds(ds):
    cols_to_keep = ['tokens', 'ner']
    for split in ds.keys():
        ds[split] = ds[split].remove_columns(
            [col for col in ds[split].column_names if col not in cols_to_keep]
            )

    return ds

In [36]:
hun_dataset = drop_cols_in_ds(hun_dataset)
eng_dataset = drop_cols_in_ds(eng_dataset)
ger_dataset = drop_cols_in_ds(ger_dataset)

Checking dataset features

In [37]:
print(f"Hungarian dataset (master) feature schema: {hun_dataset['train'].features}")
print(f"English dataset feature schema: {eng_dataset['train'].features}")
print(f"German dataset feature schema: {ger_dataset['train'].features}")

Hungarian dataset (master) feature schema: {'tokens': List(Value('string')), 'ner': List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']))}
English dataset feature schema: {'tokens': List(Value('string')), 'ner': List(Value('int64'))}
German dataset feature schema: {'tokens': List(Value('string')), 'ner': List(Value('int64'))}


In [38]:
print("Hungarian dataset NER tags sample")
[print(row, end='\n')for row in hun_dataset["train"]['ner'][:3]]
print("\n")
print("English dataset NER tags sample")
[print(row, end='\n')for row in eng_dataset["train"]['ner'][:3]]
print("\n")
print("German dataset NER tags sample")
[print(row, end='\n')for row in ger_dataset["train"]['ner'][:3]];

Hungarian dataset NER tags sample
[0, 0, 0, 3, 4, 4, 4, 0, 3, 0, 0, 0, 0, 3, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 3, 4, 0, 0, 3, 0, 0]
[0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 3, 0]
[0, 3, 4, 4, 4, 4, 0, 0, 3, 4, 4, 0, 0, 3, 4, 0, 3, 0, 0, 0, 0, 0, 3, 0, 3, 0, 0, 0]


English dataset NER tags sample
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 3, 4, 0, 0, 0, 0, 0]


German dataset NER tags sample
[1, 0, 0, 0, 3, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 1, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0]
[0, 0, 0, 0, 0, 0,

Casting master set schema to German and English datasets

In [39]:
master_features = hun_dataset["train"].features

eng_dataset = eng_dataset.cast(master_features)
ger_dataset = ger_dataset.cast(master_features)

Casting the dataset:   0%|          | 0/9482 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/559 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/546 [00:00<?, ? examples/s]

Verifying 'ner' column features accross datasets

In [40]:
print(f"Hungarian dataset (master) feature schema: {hun_dataset['train'].features['ner']}")
print(f"English dataset feature schema: {eng_dataset['train'].features['ner']}")
print(f"German dataset feature schema: {ger_dataset['train'].features['ner']}")

Hungarian dataset (master) feature schema: List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']))
English dataset feature schema: List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']))
German dataset feature schema: List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']))


Oversampling training set

In [57]:
interleaved_dataset = DatasetDict()

In [58]:
interleaved_dataset["train"] = interleave_datasets(
    [hun_dataset["train"], eng_dataset["train"], ger_dataset["train"]],
    stopping_strategy="all_exhausted" 
)

Concatenating validation and test sets

In [59]:
interleaved_dataset["validation"] = concatenate_datasets([
    hun_dataset["validation"], 
    eng_dataset["validation"], 
    ger_dataset["validation"]
])

interleaved_dataset["test"] = concatenate_datasets([
    hun_dataset["test"], 
    eng_dataset["test"], 
    ger_dataset["test"]
])

In [60]:
print("Preprocessed dataset structure:")
print(interleaved_dataset)

Preprocessed dataset structure:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner'],
        num_rows: 72000
    })
    validation: Dataset({
        features: ['tokens', 'ner'],
        num_rows: 3699
    })
    test: Dataset({
        features: ['tokens', 'ner'],
        num_rows: 7374
    })
})


In [61]:
print(f"Preprocessed training set size:    {len(interleaved_dataset['train'])}")
print(f"Preprocessed validation set size:  {len(interleaved_dataset['validation'])}")
print(f"Preprocessed test set size:        {len(interleaved_dataset['test'])}")

Preprocessed training set size:    72000
Preprocessed validation set size:  3699
Preprocessed test set size:        7374


Saving preprocessed dataset to disk

In [ ]:
interleaved_dataset.save_to_disk("data/processed/harmonized_dataset/interleaved_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/72000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3699 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7374 [00:00<?, ? examples/s]

Saving individual datasets

In [ ]:
hun_dataset.save_to_disk("data/processed/harmonized_dataset/hun_dataset")
eng_dataset.save_to_disk("data/processed/harmonized_dataset/eng_dataset")
ger_dataset.save_to_disk("data/processed/harmonized_dataset/ger_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/12282 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/940 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1728 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9482 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/559 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/546 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5100 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/24000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2200 [00:00<?, ? examples/s]